# Practical Lab 1: Streaming Data for Predictive Maintenance with Linear Regression-Based Alerts

## Project Overview

This lab extends the Data Stream Visualization Workshop by adding regression-based anomaly detection to a robot monitoring pipeline.

Historical measurements for robot **RMBR4-2** will be retrieved from **Neon PostgreSQL** to train eight univariate linear regression models, one for each of **Axes #1–#8**, using time as the input.

We will analyze regression residuals to justify two deviation thresholds, **MinC** and **MaxC**, and a minimum continuous duration, **T**. These rules will identify sustained deviations above predicted values as **Alerts** or **Errors**.

Synthetic testing data will be generated using the historical training data’s metadata and statistics, then streamed into the database. An integrated dashboard will display measurements, regression predictions, and annotated events. Detected events will be stored for review.

## Implementation Approach

Reusable processing, modeling, streaming, and detection logic will be implemented in **Python classes within separate modules**. This notebook will call those classes and document the methods, outputs, and findings.

## 1. Project Setup

Import the libraries and existing project components needed to access the historical data.

The project uses `TelemetryStore` to communicate with Neon PostgreSQL and `JOINTS` to identify the eight axis columns consistently. The notebook runs from the project root so that the existing Python modules can be imported.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from acquisition.controller_feed import JOINTS
from persistence.telemetry_store import TelemetryStore

PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / "persistence").is_dir(), (
    "Run this notebook from the project root folder."
)

print("Project folder:", PROJECT_ROOT.name)
print("Axes:", JOINTS)

Project folder: DataStreamVisualization-regression-based-anomaly-detection
Axes: ['j1', 'j2', 'j3', 'j4', 'j5', 'j6', 'j7', 'j8']


## 2. Retrieve Historical Training Data from Neon

Retrieve the persisted historical records marked `origin = 'seed'` from Neon PostgreSQL and sort them chronologically.

These database records will form the training dataset for the regression models. Synthetic testing records will remain separate from the historical training data.

Before proceeding, check that the retrieved dataset:

- Is not empty.
- Contains valid, unique timestamps.
- Has no missing values in Axes #1–#8.
- Contains only historical seed records.

Display the sample count, time range, and first five rows to inspect the retrieved data.

In [2]:
# Retrieve historical training data from the cloud database.
store = TelemetryStore()

training_df = (
    store.wide(origin="seed")
    .sort_values("sampled_at")
    .reset_index(drop=True)
)

assert not training_df.empty, "No historical training data found in Neon."
assert training_df["sampled_at"].notna().all(), "Missing timestamps found."
assert not training_df["sampled_at"].duplicated().any(), (
    "Duplicate timestamps found."
)
assert training_df[JOINTS].notna().all().all(), "Missing axis values found."
assert training_df["origin"].eq("seed").all(), (
    "Training data contains records outside the historical seed data."
)

print("Training source: Neon PostgreSQL")
print(f"Historical samples: {len(training_df):,}")
print(f"Number of axes: {len(JOINTS)}")
print("First timestamp:", training_df["sampled_at"].min())
print("Last timestamp:", training_df["sampled_at"].max())

display(training_df.head())

Training source: Neon PostgreSQL
Historical samples: 39,672
Number of axes: 8
First timestamp: 2022-10-17 12:18:23.660000+00:00
Last timestamp: 2022-10-18 10:44:58.628000+00:00


,sample_id,sampled_at,trait,state,origin,j1,j2,j3,j4,j5,j6,j7,j8
0,1,2022-10-17 12:18:23.660000+00:00,current,IDLE,seed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,2022-10-17 12:18:25.472000+00:00,current,IDLE,seed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2022-10-17 12:18:27.348000+00:00,current,IDLE,seed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,2022-10-17 12:18:29.222000+00:00,current,IDLE,seed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,2022-10-17 12:18:31.117000+00:00,current,IDLE,seed,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Findings: Database Retrieval

Successfully retrieved **39,672 historical samples** from Neon PostgreSQL, covering **October 17–18, 2022**, with measurements for all eight axes.

The validation checks passed: timestamps are present and unique, axis values have no missing entries, and all retrieved records belong to the historical training dataset (`origin = 'seed'`).

The first five samples are marked `IDLE` and contain zero values across all eight axes. We will inspect the full dataset before deciding how to handle idle periods.

## 3. Inspect Training Data and Sampling Intervals

Use the `TrainingDataInspector` class to examine the historical data retrieved from Neon.

This inspection summarizes:

- Each axis’s distribution, missing values, infinite values, and proportion of zeros.
- The proportion of samples labeled `RUNNING` or `IDLE`.
- The timing between consecutive samples and any recording gaps longer than 10 seconds.

These results will guide preprocessing, synthetic data generation, and interpretation of regression residuals. Idle samples will remain in the dataset during this inspection.

Recording gaps require special attention because missing observations cannot establish that a deviation persisted continuously.

In [3]:
from analysis.training_inspector import TrainingDataInspector

inspector = TrainingDataInspector(training_df, JOINTS)

print("AXIS STATISTICS")
display(inspector.axis_summary().round(4))

print("RECORDED OPERATING STATES")
display(inspector.activity_summary().round(2))

print("SAMPLING INTERVALS")
display(inspector.sampling_summary().round(3))

print("RECORDING GAPS LONGER THAN 10 SECONDS")
display(inspector.recording_gaps())

AXIS STATISTICS


,count,missing_count,infinite_count,mean,std,min,q1,median,q3,max,zero_pct
j1,39672.0,0,0,0.7257,2.1621,0.0,0.0,0.0,0.3127,23.6093,65.3206
j2,39672.0,0,0,3.6134,6.8800,0.0,0.0,0.0,4.2172,51.7132,65.0887
j3,39672.0,0,0,2.7103,5.1119,0.0,0.0,0.0,4.5862,41.8556,65.1442
j4,39672.0,0,0,0.6202,1.5749,0.0,0.0,0.0,0.5162,15.6663,65.5475
j5,39672.0,0,0,0.9545,2.1002,0.0,0.0,0.0,0.8001,20.7508,65.1946
j6,39672.0,0,0,0.5994,1.8155,0.0,0.0,0.0,0.3613,20.9314,65.7214
j7,39672.0,0,0,0.8701,2.1668,0.0,0.0,0.0,0.3100,8.1085,65.5626
j8,39672.0,0,0,0.1022,0.4231,0.0,0.0,0.0,0.0785,5.9056,65.4315


RECORDED OPERATING STATES


,samples,percentage
state,,
IDLE,25487,64.24
RUNNING,14185,35.76


SAMPLING INTERVALS


,interval_count,minimum_seconds,median_seconds,mean_seconds,maximum_seconds,gaps_over_10_seconds
0,39671,1.714,1.891,2.037,402.538,6


RECORDING GAPS LONGER THAN 10 SECONDS


,previous_timestamp,next_timestamp,gap_seconds
0,2022-10-17 12:27:00.597000+00:00,2022-10-17 12:33:43.135000+00:00,402.538
1,2022-10-17 13:02:12.324000+00:00,2022-10-17 13:07:44.934000+00:00,332.610
2,2022-10-17 15:13:22.042000+00:00,2022-10-17 15:13:55.155000+00:00,33.113
3,2022-10-17 15:33:24.605000+00:00,2022-10-17 15:33:57.655000+00:00,33.050
4,2022-10-17 17:03:23.722000+00:00,2022-10-17 17:03:56.535000+00:00,32.813
5,2022-10-17 20:15:10.766000+00:00,2022-10-17 20:15:44.156000+00:00,33.390


### Findings: Training Data Inspection

All eight axes contain **39,672 measurements**, with no missing or infinite values. Their measurement ranges differ substantially: Axis #2 reaches approximately **51.71**, while Axis #8 reaches approximately **5.91**.

The operating-state labels identify **25,487 idle samples (64.24%)** and **14,185 running samples (35.76%)**. These labels were derived during ingestion from whether any axis measurement was nonzero; they are not independently measured machine states.

The median sampling interval is **1.891 seconds**. Six recording gaps exceed 10 seconds, with the largest lasting **402.538 seconds**.

For the initial required Time → Axis models, we will retain both idle and running samples and preserve the original measurements. Zero readings will not be treated as missing values, and large readings will not be removed merely because they are extreme. Residual analysis will assess the limitations of fitting a straight-line baseline to data containing operating cycles and idle periods.

Elapsed time will be calculated from actual timestamps. Recording gaps will remain visible rather than being filled with invented observations.

## 4. Prepare the Time Feature

Each required regression model will use a single input: **elapsed seconds since the first historical training timestamp**. Its target will be the corresponding axis measurement.

The `TimePreprocessor` class learns this reference timestamp from training data. The same fitted reference will later be applied to synthetic testing data, so training and prediction use a consistent time coordinate.

This step preserves the original axis values and timestamp spacing. Normalization and standardization will be handled separately using parameters fitted only on historical training data.

In [4]:
from analysis.time_preprocessor import TimePreprocessor

time_preprocessor = TimePreprocessor()
prepared_training_df = time_preprocessor.fit_transform(training_df)

# Confirm that preparation preserved all original axis measurements.
pd.testing.assert_frame_equal(
    prepared_training_df[JOINTS],
    training_df[JOINTS],
)

assert prepared_training_df["elapsed_seconds"].iloc[0] == 0
assert prepared_training_df["elapsed_seconds"].is_monotonic_increasing

print("Training reference:", time_preprocessor.reference_time_)
print(f"Prepared samples: {len(prepared_training_df):,}")
print(
    "Final elapsed time:",
    f"{prepared_training_df['elapsed_seconds'].iloc[-1]:,.3f} seconds",
)
print("Original axis measurements preserved: PASS")

display(
    prepared_training_df[
        ["sampled_at", "elapsed_seconds", *JOINTS]
    ].head()
)

Training reference: 2022-10-17 12:18:23.660000+00:00
Prepared samples: 39,672
Final elapsed time: 80,794.968 seconds
Original axis measurements preserved: PASS


,sampled_at,elapsed_seconds,j1,j2,j3,j4,j5,j6,j7,j8
0,2022-10-17 12:18:23.660000+00:00,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2022-10-17 12:18:25.472000+00:00,1.812,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2022-10-17 12:18:27.348000+00:00,3.688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2022-10-17 12:18:29.222000+00:00,5.562,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2022-10-17 12:18:31.117000+00:00,7.457,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Findings: Time-Feature Preparation

Successfully created `elapsed_seconds` for all **39,672 historical samples**, using **2022-10-17 12:18:23.660 UTC** as the fixed training reference.

Elapsed time ranges from **0 to 80,794.968 seconds**. Checks confirmed chronological ordering and preservation of the original axis measurements.

The fitted time preprocessor will use this same reference when transforming synthetic testing data.

## 5. Fit Normalization and Standardization on Training Data

The axis measurements have different ranges and variability. We will create two separate scaled versions using the historical training data retrieved from Neon:

- **Min–Max normalization:** `(value − training minimum) / training range`.
- **Z-score standardization:** `(value − training mean) / training standard deviation`.

The `AxisScaler` class stores the training parameters and applies them without modification to subsequent datasets. The Z-score calculation uses population standard deviation (`ddof=0`); the earlier descriptive statistics used sample standard deviation (`ddof=1`).

The original measurements and elapsed-time feature remain available. Normalized and standardized values are dimensionless. Predictions and alert deviations will be interpreted in the original measurement units.

Testing values may fall outside the training Min–Max interval of `[0, 1]`. We will not clip them, because doing so could hide unusually large deviations. Testing data is also not expected to have a Z-score mean of zero or standard deviation of one when transformed using training parameters.

In [5]:
import numpy as np

from analysis.axis_scaler import AxisScaler

axis_scaler = AxisScaler(JOINTS)
axis_scaler.fit(prepared_training_df)

normalized_training_df = axis_scaler.transform(
    prepared_training_df,
    method="minmax",
)

standardized_training_df = axis_scaler.transform(
    prepared_training_df,
    method="zscore",
)

print("PARAMETERS FITTED ONLY ON HISTORICAL TRAINING DATA")
display(axis_scaler.parameters_.round(6))

scaling_checks = pd.DataFrame({
    "normalized_min": normalized_training_df[JOINTS].min(),
    "normalized_max": normalized_training_df[JOINTS].max(),
    "zscore_mean": standardized_training_df[JOINTS].mean(),
    "zscore_std": standardized_training_df[JOINTS].std(ddof=0),
})

print("TRAINING SCALING CHECKS")
display(scaling_checks.round(6))

# Confirm both representations recover the original axis values.
for method, scaled_frame in [
    ("minmax", normalized_training_df),
    ("zscore", standardized_training_df),
]:
    restored = axis_scaler.inverse_transform(scaled_frame, method)

    np.testing.assert_allclose(
        restored[JOINTS].to_numpy(),
        prepared_training_df[JOINTS].to_numpy(),
        rtol=1e-10,
        atol=1e-10,
    )

    pd.testing.assert_series_equal(
        scaled_frame["elapsed_seconds"],
        prepared_training_df["elapsed_seconds"],
    )

print("Both inverse transformations recover original values: PASS")
print("Elapsed-time feature preserved in both representations: PASS")

PARAMETERS FITTED ONLY ON HISTORICAL TRAINING DATA


,minimum,maximum,mean,std,range
j1,0.0,23.60930,0.725743,2.162093,23.60930
j2,0.0,51.71323,3.613374,6.879875,51.71323
j3,0.0,41.85556,2.710336,5.111836,41.85556
j4,0.0,15.66630,0.620222,1.574877,15.66630
j5,0.0,20.75076,0.954521,2.100159,20.75076
j6,0.0,20.93142,0.599427,1.815475,20.93142
j7,0.0,8.10848,0.870145,2.166783,8.10848
j8,0.0,5.90564,0.102214,0.423070,5.90564


TRAINING SCALING CHECKS


,normalized_min,normalized_max,zscore_mean,zscore_std
j1,0.0,1.0,0.0,1.0
j2,0.0,1.0,0.0,1.0
j3,0.0,1.0,-0.0,1.0
j4,0.0,1.0,0.0,1.0
j5,0.0,1.0,-0.0,1.0
j6,0.0,1.0,-0.0,1.0
j7,0.0,1.0,-0.0,1.0
j8,0.0,1.0,0.0,1.0


Both inverse transformations recover original values: PASS
Elapsed-time feature preserved in both representations: PASS


### Findings: Training Data Scaling

Min–Max normalization and Z-score standardization were successfully applied to all eight axes using parameters fitted exclusively on historical data retrieved from Neon.

Each normalized training axis spans **0–1**. Each standardized training axis has a mean approximately equal to **0** and a population standard deviation approximately equal to **1**.

Both inverse transformations recovered the original measurements within numerical tolerance. The elapsed-time feature was preserved.

These fitted scaling parameters will be reused for synthetic testing data without refitting. This completes training-scaler preparation; testing-data transformation remains pending.